# B5. Word2Vec and Sentence Embeddings

*ML and NLP course - Data Trainers LLC - Axel Sirota*

## The story

You are the ML/NLP person on the customer-support platform from Part A. So far you have
called `pipeline()` and a sentence embedder as black boxes. Now the team wants two things:
a semantic search over the support knowledge base (so agents stop missing relevant articles),
and classifier features that do not require a GPU on every single request.

Both come from the same idea: text becomes a **vector**, and vectors live in a space where
**direction and distance carry meaning**. In B4 you learned tensors and autograd. Here you
learn the geometry that the rest of the course classifies, searches, and fine-tunes.

We will NOT train embeddings from scratch. We load **pretrained** ones, because that is what
you do in real life, and explore the geometry.

## What you will be able to do

- Load pretrained word vectors and query them: nearest neighbors, cosine similarity, analogies.
- Project word vectors to 2D and read the clusters.
- Explain why averaging word vectors is a weak sentence representation.
- Encode sentences with a pretrained transformer and build a working semantic search.
- Build a document vector (mean-pooled word vectors): the exact feature the B7 MLP will use.

## Prerequisites

- B4: tensors, dtype/shape, matrix ops, broadcasting, autograd.
- Comfort with numpy arrays and cosine similarity as "dot product over norms".

## Session format

Theory -> Demo -> Lab, three labs. Runs on Colab CPU; a GPU only speeds up the sentence
encoder. About 60-90 minutes.

## Section 0. Environment Setup

We install a small, pinned stack, import libraries, fix the random seed, and detect the GPU.

Two pins matter and are easy to get wrong:

- `numpy<2`. Colab now ships numpy 2.x, but `gensim` 4.3 requires numpy < 2. If numpy 2 is
  already imported, you may need to restart the runtime after the install cell (Runtime ->
  Restart session) and run again.
- `scipy<1.13`. gensim 4.3 imports `scipy.linalg.triu`, which scipy removed in 1.13. Without
  this pin you get `ImportError: cannot import name 'triu' from 'scipy.linalg'`.

In [ ]:
# Install required packages (run this first on Colab).
# Pins explained in the markdown above:
#   numpy<2          gensim 4.3 needs it; Colab ships numpy 2.x by default.
#   scipy<1.13       gensim 4.3 imports scipy.linalg.triu, removed in scipy 1.13.
#   gensim==4.3.3    stable; brings gensim.downloader for pretrained vectors.
#   sentence-transformers pinned to a 3.x line that works with transformers 4.57.x.
#   transformers==4.57.* (NOT 5.x: 5.x forces numpy 2 and drops pipelines we use later).
!pip install -q "numpy<2" "scipy<1.13" "gensim==4.3.3" \
    "sentence-transformers==3.4.1" "transformers==4.57.1" \
    "datasets>=2.19,<3" "scikit-learn>=1.3" "matplotlib>=3.7" "pandas>=2.0"

# If you see a numpy-2 ImportError after this cell, restart the runtime and re-run from here.
print("Install done. If numpy was already 2.x, restart the runtime now and re-run.")

# Download the small English spaCy pipeline (model weights, separate from pip).
!python -m spacy download en_core_web_sm

# TextBlob/NLTK tokenizer data (punkt_tab) for sentence/word tokenization.
import nltk
nltk.download('punkt_tab')


In [ ]:
# Core imports - grouped: data -> embeddings -> similarity -> viz.
import warnings
import numpy as np
import pandas as pd

import torch
import gensim.downloader as api               # pretrained word vectors, no training
from sentence_transformers import SentenceTransformer, util

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Versions - useful when a student reports a bug.
print("numpy   :", np.__version__)            # expect 1.26.x (NOT 2.x)
import gensim, transformers
print("gensim  :", gensim.__version__)        # expect 4.3.3
print("transformers:", transformers.__version__)

# Reproducibility: one seed for every RNG we touch.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device: GPU if present (only the sentence encoder benefits). Same idiom as B4.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## What we are building today

Three concrete artifacts, all from pretrained models:

1. **A word-vector explorer.** Load 400,000 pretrained word vectors and ask: what is near
   "king"? what is "paris - france + italy"? Plot a handful in 2D.
2. **A sentence encoder.** Turn whole sentences into vectors that respect meaning (not just
   word overlap), and measure their similarity against the STS-B benchmark.
3. **A semantic search engine.** Encode a corpus once, then answer free-text queries by
   nearest-neighbor in vector space. This is the retrieval step a RAG chatbot runs before it
   answers.

Everything is geometry. By the end, "embedding" should feel like "a point in space," and
"similar" should feel like "a small angle between two points."

[View diagram](diagrams/word2vec-sentence-embeddings/query-word-space.mmd)

The diagram shows how one shared geometric space underlies all three artifacts we build today.

In [ ]:
# Load pretrained GloVe vectors via gensim's downloader. First call downloads ~130MB and
# caches it under ~/gensim-data, so re-runs in the same session are instant.
# glove-wiki-gigaword-100: 400,000 words, 100 dimensions, trained on Wikipedia + Gigaword.
print("Loading pretrained word vectors (first run downloads ~130MB)...")
wv = api.load("glove-wiki-gigaword-100")      # returns a gensim KeyedVectors object

print("Vocabulary size :", len(wv))                       # ~400,000
print("Vector size     :", wv.vector_size)                # 100
print("Vector for 'coffee' (first 5 dims):", wv["coffee"][:5])

## Section 1. Why Embeddings?

### The one-hot problem

The simplest way to represent a word is a one-hot vector: a column index in a vocabulary.

```python
king  = [1, 0, 0, 0, ...]
queen = [0, 1, 0, 0, ...]
dog   = [0, 0, 1, 0, ...]
```

The cosine similarity between any two distinct one-hot vectors is exactly 0. The
representation claims `king` is as unrelated to `queen` as it is to `dog`. That is a hard
ceiling on what any downstream model can recover. Dense embeddings lift the ceiling: similar
words get similar vectors. The next cell shows the failure numerically.

[View diagram](diagrams/word2vec-sentence-embeddings/one-hot-vs-dense.mmd)

The diagram contrasts orthogonal one-hot vectors (all pairs at cosine 0) with dense vectors where related words cluster.

**Figure: One-hot vs dense - why orthogonal one-hot vectors cannot encode word similarity.**

![One-hot vs dense embeddings](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/word2vec-sentence-embeddings/one-hot-vs-dense.png)


In [ ]:
# Tiny demo: one-hot vectors are mutually orthogonal, so every pair has cosine 0.
vocab_toy = ["king", "queen", "dog", "car"]
one_hot = np.eye(len(vocab_toy))              # 4x4 identity = 4 one-hot rows

sims = cosine_similarity(one_hot)             # pairwise cosine over the rows
print("Cosine similarity of one-hot vectors:")
print(pd.DataFrame(sims, index=vocab_toy, columns=vocab_toy))
print("\nEvery off-diagonal entry is 0: king = queen = dog = car, as far as the model knows.")

### Querying a pretrained space

We already loaded `wv` in Section 0: 400,000 words, each a 100-dimensional dense vector,
learned by GloVe on Wikipedia. Unlike one-hot, these vectors put related words near each
other. gensim's `KeyedVectors` gives us the query API:

```python
wv.most_similar("coffee", topn=5)     # 5 nearest neighbors by cosine
wv.similarity("coffee", "tea")        # cosine between two words
"coffee" in wv.key_to_index           # membership test, avoids KeyError
```

Two things to remember:

- The vocabulary is lowercase. Query `"paris"`, not `"Paris"`.
- A word not in the vocabulary raises `KeyError`. Always guard with `in wv.key_to_index`
  (equivalently `in wv`) before you index.

**Figure: Querying the pretrained word space - guard for OOV, then read neighbors and cosines.**

![Querying the pretrained word space](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/word2vec-sentence-embeddings/query-word-space.png)


In [ ]:
# Demo: nearest neighbors and pairwise cosine on the pretrained vectors.
for anchor in ["coffee", "king", "python"]:
    print(f"\nTop-5 neighbors of '{anchor}':")
    for word, score in wv.most_similar(anchor, topn=5):    # returns (word, cosine) pairs
        print(f"  {word:15s}  cos={score:.3f}")

# Direct pairwise cosine between two words.
print("\nsimilarity(coffee, tea)  =", round(wv.similarity("coffee", "tea"), 3))
print("similarity(coffee, car)  =", round(wv.similarity("coffee", "car"), 3))
# Note: 'python' pulls BOTH the language and the snake. Static vectors have ONE vector per
# word, so the two senses are blended. That polysemy limit is why Part C reaches for
# contextual transformers.

### Vector arithmetic: king - man + woman ~ queen

The poster-child result of word embeddings: semantic relationships show up as directions in
the space. "Man -> woman" is roughly the same direction as "king -> queen", so

```python
wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
```

returns `queen` near the top. gensim computes `king - man + woman` and finds the nearest
vector, automatically EXCLUDING the three input words (otherwise it would just return `king`).

[View diagram](diagrams/word2vec-sentence-embeddings/analogy-arithmetic.mmd)

The diagram shows the parallelogram of offsets that makes king - man + woman land near queen.

### Honesty check

This works on Wikipedia-scale GloVe, but it is more fragile than the hype suggests. Many
"analogies" only work because the input words are excluded. And the same geometry encodes
social bias: `doctor - man + woman` can return `nurse`, because the training text carried that
stereotype. Embeddings learn the corpus, biases and all. We will look at one such case below.

**Figure: Analogy arithmetic - king minus man plus woman, and the bias the same geometry encodes.**

![Analogy vector arithmetic](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/word2vec-sentence-embeddings/analogy-arithmetic.png)


In [ ]:
# Demo: vector arithmetic. most_similar(positive=..., negative=...) does king - man + woman.
def analogy(a, b, c, topn=3):
    """a is to b as c is to ?  ->  b - a + c.  Returns top matches, input words excluded."""
    return wv.most_similar(positive=[b, c], negative=[a], topn=topn)

print("man : king  ::  woman : ?")
print(analogy("man", "king", "woman"))          # expect queen near the top

print("\nfrance : paris  ::  italy : ?")
print(analogy("france", "paris", "italy"))      # expect rome

print("\nwalk : walked  ::  run : ?")
print(analogy("walk", "walked", "run"))         # expect ran (morphology direction)

# The honesty check: the geometry also encodes bias from the training corpus.
print("\nman : doctor  ::  woman : ?  (watch for stereotype)")
print(analogy("man", "doctor", "woman"))

In [ ]:
# ============================================================
# Lab 1. Probe the word-vector space (15 min) - SOLUTION
# ============================================================
# You have `wv` (pretrained GloVe, 100d) and the `analogy()` helper from the demo.
#
# Task A: For each anchor word, print its top-5 nearest neighbors. GUARD against words that
#         are not in the vocabulary so you never hit a KeyError.
# Task B: Complete one analogy of your own choosing using the analogy() helper.
#
# Reminder: the vocabulary is lowercase; the gensim membership test avoids a KeyError.

anchors = ["doctor", "guitar", "pizza", "rocket", "winter"]

# Task A: nearest neighbors with an OOV guard.
for anchor in anchors:
    # 1. Membership test. `in wv.key_to_index` is the gensim-4 way to check vocabulary
    #    WITHOUT triggering a KeyError. `anchor in wv` works too; we use the explicit form
    #    so the intent is obvious. Why: indexing wv[oov_word] raises, so we must guard first.
    in_vocab = anchor in wv.key_to_index
    if not in_vocab:
        print(f"'{anchor}' not in vocab, skipping")
        continue
    # 2. Top-5 neighbors. most_similar returns a list of (word, cosine) pairs already sorted
    #    high-to-low. topn=5 caps the list at five. Common mistake: forgetting topn and
    #    printing the default 10.
    neighbors = wv.most_similar(anchor, topn=5)
    print(f"\nTop-5 neighbors of '{anchor}':")
    for word, score in neighbors:
        print(f"  {word:15s}  cos={score:.3f}")

# Task B: one analogy of your choice. Pick a, b, c so that "a is to b as c is to ?".
# Here: tokyo is to japan as paris is to ? (expect france). The helper does b - a + c.
my_analogy = analogy("tokyo", "japan", "paris")

# ---- silent rescue (lets a straight run reach the verification below) ----
# In the exercise this fills in blank tasks; here the lab is solved, so it never fires.
if "in_vocab" not in dir() or in_vocab is None:
    in_vocab = anchors[0] in wv.key_to_index
if "neighbors" not in dir() or neighbors is None:
    neighbors = wv.most_similar(anchors[0], topn=5)
if my_analogy is None:
    my_analogy = analogy(anchors[0], anchors[0], anchors[0])
print("\nMy analogy result:", my_analogy)

<details>
<summary>Stuck on Lab 1? Reveal the safety-net</summary>

```python
in_vocab = anchor in wv.key_to_index
neighbors = wv.most_similar(anchor, topn=5)
my_analogy = analogy("paris", "france", "rome")
```
</details>


In [ ]:
# ---- Lab 1 verification (provided) ----
assert in_vocab is not None, "Task A step 1: set in_vocab to a membership test."
assert isinstance(neighbors, list) and len(neighbors) == 5, \
    "Task A step 2: neighbors should be a list of 5 (word, score) pairs."
assert all(isinstance(w, str) and isinstance(s, float) for w, s in neighbors), \
    "neighbors entries should be (str, float)."
assert my_analogy is not None and len(my_analogy) >= 1, \
    "Task B: my_analogy should be the result of analogy(a, b, c)."
print("Lab 1 passed. You can query a 400k-word space by meaning, with an OOV guard.")

In [ ]:
# Demo: project a chosen set of words to 2D with PCA so we can eyeball the geometry.
# PCA (not t-SNE) on purpose: it is deterministic, fast, and needs no perplexity tuning,
# which makes it the right tool for a small, hand-picked word set.
words_2d = [
    "king", "queen", "man", "woman",          # gender / royalty
    "paris", "london", "rome", "berlin",      # capitals
    "coffee", "tea", "water", "juice",        # drinks
]
vectors_2d = np.stack([wv[w] for w in words_2d])          # (12, 100)

coords = PCA(n_components=2, random_state=SEED).fit_transform(vectors_2d)   # (12, 2)

plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=40)
for (x, y), w in zip(coords, words_2d):
    plt.annotate(w, (x, y), fontsize=11, xytext=(4, 4), textcoords="offset points")
plt.title("PCA of GloVe vectors (100d -> 2d). Watch the drinks / capitals / royalty groups.")
plt.grid(True, alpha=0.3)
plt.show()
# 2D throws away 98 dimensions, so do not over-read it. But related words should still cluster.

## Section 2. From Words to Sentences

Almost every real product works at the sentence or document level: FAQ matching, duplicate
detection, semantic search, chatbot intent, RAG. So we need one vector per SENTENCE, not per
word.

### The naive baseline: average the word vectors

The simplest sentence vector is the mean of its word vectors:

```python
def mean_vector(sentence):
    vecs = [wv[w] for w in sentence.lower().split() if w in wv]
    return np.mean(vecs, axis=0)
```

It is a real baseline, and sometimes a strong one. But it has two fatal flaws we can see in
one experiment: it ignores word ORDER ("dog bites man" == "man bites dog"), and it barely
distinguishes ANTONYMS ("I love it" vs "I hate it" share every word but one). The next cell
makes both failures concrete.

**Figure: The mean-of-word-vectors baseline and its two failure modes.**

![Mean-of-word-vectors baseline and failures](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/word2vec-sentence-embeddings/mean-vector-failures.png)


In [ ]:
# Demo: build the mean-of-word-vectors baseline using wv, then expose its two failure modes.
def mean_vector(sentence, dim=100):
    """Mean of in-vocab word vectors. Returns zeros if no token is known."""
    vecs = [wv[w] for w in sentence.lower().split() if w in wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim, dtype=np.float32)

def cos(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))

pairs = [
    ("dog bites man",          "man bites dog"),           # word order
    ("i love it",              "i hate it"),               # antonym
    ("the food was excellent", "the meal was outstanding"),# true paraphrase
    ("the food was excellent", "my car broke down"),       # unrelated
]
print(f"{'Sentence A':<26}{'Sentence B':<26}{'mean-vec cos':>14}")
print("-" * 66)
for a, b in pairs:
    print(f"{a:<26}{b:<26}{cos(mean_vector(a), mean_vector(b)):>14.3f}")
# Expect: order pair ~1.00 (identical), antonym pair ~0.95 (no discrimination). Broken.

## Sentence-BERT (SBERT)

[Reimers and Gurevych, EMNLP 2019](https://arxiv.org/abs/1908.10084) introduced Sentence-BERT:
take a pretrained BERT, mean-pool its token vectors, then fine-tune the whole stack on labeled
sentence pairs (NLI, STS) with a siamese setup. The result is an encoder where ONE forward
pass turns a sentence into a fixed-size vector, and cosine similarity in that space tracks
SEMANTIC similarity, with word order and composition respected.

We use `all-MiniLM-L6-v2`: 6 layers, 384-dimensional output, ~22M parameters, around 14,000
sentences/second on a CPU. This is the SAME model you met in Part A as the embedder, so the
text-to-vector tool you called as a black box is now something you understand.

[View diagram](diagrams/word2vec-sentence-embeddings/sbert-encode.mmd)

The diagram shows the shared-weight siamese encoder that pools tokens into one sentence vector per input.

> Caveat: SBERT is paraphrase-aware, not polarity-aware. It captures topical similarity very
> well; for sentiment you still want a sentiment-tuned classifier (that is Part C).

**Figure: SBERT - one forward pass and a mean-pool turn a sentence into a 384-dim vector.**

![SBERT sentence encoding](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/word2vec-sentence-embeddings/sbert-encode.png)


In [ ]:
# Load the SAME sentence encoder used in Part A. First call downloads ~80MB and caches it.
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=str(device))
print("Embedding dimension:", embedder.get_sentence_embedding_dimension())   # 384

# Encode all sentences in one batched call: list in, ndarray out (one row per sentence).
sents_A = [a for a, _ in pairs]
sents_B = [b for _, b in pairs]
emb_A = embedder.encode(sents_A, convert_to_numpy=True)   # (4, 384)
emb_B = embedder.encode(sents_B, convert_to_numpy=True)

# Re-run the four pairs: mean-vec baseline vs SBERT, side by side.
print(f"{'Sentence A':<26}{'Sentence B':<26}{'mean-vec':>10}{'SBERT':>10}")
print("-" * 72)
for i, (a, b) in enumerate(pairs):
    base = cos(mean_vector(a), mean_vector(b))
    sbert = float(util.cos_sim(emb_A[i], emb_B[i]))       # util.cos_sim handles L2 norm
    print(f"{a:<26}{b:<26}{base:>10.3f}{sbert:>10.3f}")
# Expect the antonym pair to drop sharply under SBERT, and the true paraphrase to stay high.

### Lab 2. Score real sentence similarity against STS-B (15 min)

STS-B (Semantic Textual Similarity Benchmark) is a standard set of sentence pairs, each with a
human similarity score from 0 (unrelated) to 5 (equivalent). We will encode the pairs, compute
cosine similarity, and check whether our cosines RANK the pairs the way humans did.

**Why Spearman and not Pearson.** Pearson measures whether two quantities move together on a
straight LINE, so it would punish us for the fact that cosine lives in `[-1, 1]` while the gold
score lives in `[0, 5]` - two different scales with a non-linear relationship. We do not care
about matching the exact numbers; we care that the pair humans rated most similar is the pair
our encoder scores highest, and so on down the list. Spearman is exactly that: it correlates
the RANKS instead of the raw values, so any order-preserving (monotonic) relationship scores a
perfect 1.0. That is the right yardstick for a similarity model, and it is the metric the STS-B
leaderboard reports.

Steps:

1. Load STS-B: `load_dataset("glue", "stsb", split="validation")`. Take the first 200 rows.
   Fields are `sentence1`, `sentence2`, `label` (the 0-5 gold score).
2. Encode `sentence1` and `sentence2` with `embedder`.
3. For each pair, compute SBERT cosine into a list `preds`.
4. Compute the Spearman correlation between `preds` and the gold `labels`.

A well-behaved encoder lands around 0.8 Spearman on STS-B. Higher is better.

In [ ]:
# ============================================================
# Lab 2. Sentence similarity vs STS-B gold scores (15 min) - SOLUTION
# ============================================================
from datasets import load_dataset
from scipy.stats import spearmanr

# Provided: load 200 validation pairs from STS-B.
stsb = load_dataset("glue", "stsb", split="validation").select(range(200))
s1 = stsb["sentence1"]
s2 = stsb["sentence2"]
gold = stsb["label"]            # human scores in [0, 5]

# 1. Encode each sentence column into a numpy array. encode takes a LIST of strings and,
#    with convert_to_numpy=True, returns an (n, 384) ndarray (one row per sentence). Why a
#    batched call: it is far faster than encoding sentences one at a time in a loop.
emb1 = embedder.encode(s1, convert_to_numpy=True)   # (len(s1), 384)
emb2 = embedder.encode(s2, convert_to_numpy=True)   # (len(s2), 384)

# 2. One cosine per row pair. util.cos_sim(a, b) L2-normalizes and returns a 1x1 tensor for
#    two single vectors, so we pull the scalar out with .item() before appending. Common
#    mistake: appending the tensor itself (then spearmanr gets tensors, not numbers).
preds = []
for i in range(len(emb1)):
    preds.append(util.cos_sim(emb1[i], emb2[i]).item())

# 3. Spearman RANK correlation. spearmanr returns a result object; the coefficient is in its
#    .correlation attribute (also unpackable as the first tuple element). We use Spearman, not
#    Pearson, because we care whether the ORDER of our cosines matches the order of human
#    scores, not the exact values.
rho = spearmanr(preds, gold).correlation

# ---- silent rescue (fills blanks in the exercise; here the lab is solved, so it no-ops) ----
if emb1 is None:
    emb1 = embedder.encode(s1, convert_to_numpy=True)
if emb2 is None:
    emb2 = embedder.encode(s2, convert_to_numpy=True)
if preds is None:
    preds = [util.cos_sim(emb1[i], emb2[i]).item() for i in range(len(emb1))]
if rho is None:
    rho = spearmanr(preds, gold).correlation

# ---- verification (provided): expected sizes derive from the data, not magic numbers ----
n = len(s1)
dim = embedder.get_sentence_embedding_dimension()
assert emb1 is not None and emb1.shape == (n, dim), f"Step 1: emb1 should be ({n}, {dim})."
assert isinstance(preds, list) and len(preds) == n, f"Step 2: preds should be length {n}."
assert rho is not None and 0.6 < rho < 1.0, \
    f"Step 3: Spearman should be ~0.8 for a good encoder; got {rho}."
print(f"Lab 2 passed. SBERT cosine ranks sentence pairs like humans (Spearman={rho:.3f}).")

<details>
<summary>Stuck on Lab 2? Reveal the safety-net</summary>

```python
emb1 = embedder.encode(s1, convert_to_numpy=True)
emb2 = embedder.encode(s2, convert_to_numpy=True)
preds = [util.cos_sim(emb1[i], emb2[i]).item() for i in range(len(emb1))]
rho = spearmanr(preds, gold).correlation
```
</details>


In [ ]:
# Demo: a working semantic search engine. Encode a corpus ONCE (offline), then answer
# free-text queries by nearest-neighbor in vector space (online). This is the retrieval
# step a RAG chatbot runs before it answers.
corpus = [
    "Tracking a missing shipment",
    "How to reset your account password",
    "Refund policy for damaged items",
    "Updating your billing address",
    "Cancelling a subscription before renewal",
    "Why was my payment declined",
    "Changing the email on your account",
    "Estimated delivery times by region",
]
# Offline step: precompute corpus embeddings once. In production you cache these.
corpus_emb = embedder.encode(corpus, convert_to_tensor=True)     # (8, 384) on `device`

def semantic_search(query, top_k=3):
    """Encode the query, score against the cached corpus, return top-k (score, text)."""
    q_emb = embedder.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(q_emb, corpus_emb, top_k=top_k)[0]  # list of {corpus_id, score}
    return [(h["score"], corpus[h["corpus_id"]]) for h in hits]

# Online step: a query with NO literal word overlap with the right answer.
for score, text in semantic_search("my package never arrived"):
    print(f"  [{score:.3f}] {text}")
# Top hit should be 'Tracking a missing shipment' though they share zero words. Keyword
# search would miss it; geometry finds it.

**Figure: Semantic search - encode the corpus once offline, encode the query online, retrieve by cosine.**

![Semantic search offline and online](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/word2vec-sentence-embeddings/semantic-search-offline-online.png)


In [ ]:
# ============================================================
# Lab 3. Build your own semantic search (15 min) - SOLUTION
# ============================================================
# You have `embedder` and the `corpus` / `corpus_emb` from the demo.
# Build a search that returns the single best match and its score for any query.
#
# 1. Encode the query into a tensor.
# 2. Score it against corpus_emb with util.semantic_search, asking for the top 1 hit.
# 3. Return (score, text) for that best hit.

def best_match(query):
    # 1. Encode the query exactly like the demo did a single query: convert_to_tensor=True so
    #    it lives on the same device/dtype as corpus_emb. A single string in -> one embedding.
    q_emb = embedder.encode(query, convert_to_tensor=True)
    # 2. util.semantic_search(query_emb, corpus_emb, top_k=1) returns a list-of-lists: one
    #    inner list per query. We passed one query, so the hits for it are at index [0]. Each
    #    hit is a dict {'corpus_id': int, 'score': float}, already sorted best-first.
    hits = util.semantic_search(q_emb, corpus_emb, top_k=1)
    # 3. The single best hit is the first dict of the first (and only) query's hit list.
    #    Common mistake: forgetting the outer [0] and indexing the list-of-lists directly.
    top = hits[0][0]

    # ---- silent rescue (fills blanks in the exercise; here it never fires) ----
    if q_emb is None:
        q_emb = embedder.encode(query, convert_to_tensor=True)
    if hits is None:
        hits = util.semantic_search(q_emb, corpus_emb, top_k=1)
    if top is None:
        top = hits[0][0]
    return (top["score"], corpus[top["corpus_id"]])

# ---- verification (provided) ----
score, text = best_match("I forgot my login details")
assert isinstance(score, float), "best_match should return a float score first."
assert text == "How to reset your account password", \
    f"Expected the password-reset article, got: {text!r}"
print(f"Lab 3 passed. Best match: [{score:.3f}] {text}")

<details>
<summary>Stuck on Lab 3? Reveal the safety-net</summary>

```python
q_emb = embedder.encode(query, convert_to_tensor=True)
hits = util.semantic_search(q_emb, corpus_emb, top_k=1)
top = hits[0][0]
```
</details>


## Homework and Wrap-up

### What you can now do

- Query a 400,000-word space by meaning: neighbors, cosine, analogies (with an honest view of
  fragility and bias).
- Encode sentences with a pretrained transformer and measure quality against STS-B (Spearman).
- Ship a semantic search engine in under 50 lines that beats keyword matching.

### Homework Extension (async, deeper, builds B7's input)

This is the bridge to the B7 STOPPER. There, you train an MLP whose INPUT is a document vector.
Here you build that exact feature.

1. **`doc_vector(text)`**: mean-pool the word2vec vectors of a text into one fixed-size 100-d
   vector, skipping out-of-vocab tokens (reuse the `mean_vector` idea). This is the standard
   "average word2vec" document feature.
2. Load ~500 short labeled texts (for example `load_dataset("glue", "sst2", split="train")`,
   fields `sentence` and `label`). Build a feature matrix `X` of shape `(500, 100)` by applying
   `doc_vector` to each, and a label array `y`.
3. Sanity-check: fit a plain `LogisticRegression` on `X, y` and report accuracy. That number is
   the BASELINE the B7 MLP must beat. Save `X` and `y` (`np.save`) so B7 can load them.

### Stretch options (pick one)

- **Asymmetric search**: re-run Lab 3 with `multi-qa-MiniLM-L6-cos-v1` (built for short query /
  long answer) and compare hits to `all-MiniLM-L6-v2` on a query like "what is the refund
  window". Symmetric vs asymmetric models retrieve differently.
- **Scaling note**: brute-force cosine is fine up to ~1M vectors. Read the FAISS index docs and
  describe (in a markdown cell) when you would switch to an IVF or HNSW index and why.
- **Clustering**: run `KMeans(n_clusters=4)` on `corpus_emb` and print each cluster's members.
  Does unsupervised structure match the article topics?

### One-line bridge to B6

Text is now a fixed-size vector. Next we need a model that maps those vectors to a label.
B6 builds the PyTorch neural-net machinery (`nn.Module`, loss, optimizer, DataLoader); B7
points that machinery at these embeddings and beats the baseline you just measured.

### Resources

- gensim KeyedVectors: https://radimrehurek.com/gensim/models/keyedvectors.html
- Sentence Transformers semantic search: https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html
- SBERT paper (Reimers and Gurevych, 2019): https://arxiv.org/abs/1908.10084